# Marine Audio Dataset Processing

This notebook:
1. **Explores** the dataset — how many audio files, how many have transcripts (JSON), and basic stats.
2. **Segments** audio files to keep only the non-speech portions (noise between transcript timestamps).
3. **Saves** segmented files to `output_segmented/` mirroring the original directory structure.

## 0. Imports & Config

In [ ]:
import os
import json
import sys
from pathlib import Path
from collections import defaultdict

import pandas as pd
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
from tqdm.notebook import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR   = Path("Marine_audio_with_transcripts")
OUTPUT_DIR = Path("output_segmented")

# Minimum noise segment length to save (ms). Shorter pieces are discarded.
MIN_SEGMENT_MS = 500

# Silence-removal parameters
# A stretch of audio is considered silent if it stays below SILENCE_THRESH_DB
# for at least SILENCE_MIN_LEN_MS milliseconds.  Such stretches are dropped.
SILENCE_THRESH_DB  = -50   # dBFS
SILENCE_MIN_LEN_MS = 1000  # 1 second of continuous silence → discard

print(f"Data root  : {DATA_DIR.resolve()}")
print(f"Output root: {OUTPUT_DIR.resolve()}")
print(f"Silence threshold : {SILENCE_THRESH_DB} dBFS  (min duration {SILENCE_MIN_LEN_MS} ms)")

---
## 1. Data Exploration

In [ ]:
# Scan every sub-directory — only keep files that have a valid (non-empty) transcript
rows = []

for station_dir in sorted(DATA_DIR.iterdir()):
    if not station_dir.is_dir():
        continue
    station = station_dir.name

    mp3_files  = set(p.stem for p in station_dir.glob("*.mp3"))
    json_files = set(p.stem for p in station_dir.glob("*.json"))

    total_mp3   = len(mp3_files)
    skipped     = 0

    for stem in sorted(mp3_files):
        json_path = station_dir / f"{stem}.json"

        # Only include files whose JSON exists and contains at least one segment
        if stem not in json_files:
            skipped += 1
            continue
        try:
            data = json.loads(json_path.read_text())
            if not isinstance(data, list) or len(data) == 0:
                skipped += 1
                continue
            n_segments = len(data)
        except Exception:
            skipped += 1
            continue

        mp3_path = station_dir / f"{stem}.mp3"
        size_kb  = mp3_path.stat().st_size / 1024

        rows.append({
            "station"   : station,
            "stem"      : stem,
            "n_segments": n_segments,
            "size_kb"   : round(size_kb, 1),
            "mp3_path"  : mp3_path,
            "json_path" : json_path,
        })

    print(f"Station {station:>6}: {total_mp3:3d} total MP3s, "
          f"{total_mp3 - skipped:3d} with valid transcript, "
          f"{skipped:3d} skipped (no/empty JSON)")

df = pd.DataFrame(rows)
print(f"\n── Summary ──────────────────────────────")
print(f"Files with valid transcript (will be processed): {len(df)}")
print(f"Total speech segments across all files         : {df['n_segments'].sum()}")
df.head(10)


In [ ]:
# Per-station breakdown (all rows already have valid transcripts)
summary = df.groupby("station").agg(
    files_with_transcript = ("stem",       "count"),
    total_size_mb         = ("size_kb",    lambda x: round(x.sum()/1024, 2)),
    total_segments        = ("n_segments", "sum"),
    avg_segments_per_file = ("n_segments", lambda x: round(x.mean(), 1)),
).reset_index()

print("Per-station summary (transcript-paired files only):")
print(summary.to_string(index=False))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Marine Audio Dataset (transcript-paired files only)", fontsize=14, fontweight="bold")

# 1. Files per station
axes[0].bar(summary["station"], summary["files_with_transcript"], color="seagreen")
axes[0].set_title("Files with transcript per station")
axes[0].set_xlabel("Station ID")
axes[0].set_ylabel("# Files")
axes[0].tick_params(axis='x', rotation=45)

# 2. Average speech segments per file
axes[1].bar(summary["station"], summary["avg_segments_per_file"], color="darkorange")
axes[1].axhline(summary["avg_segments_per_file"].mean(), color="red", linestyle="--",
                label=f'Mean {summary["avg_segments_per_file"].mean():.1f}')
axes[1].set_title("Avg speech segments per file")
axes[1].set_xlabel("Station ID")
axes[1].set_ylabel("Segments")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# 3. Total audio size (MB)
axes[2].bar(summary["station"], summary["total_size_mb"], color="mediumpurple")
axes[2].set_title("Total audio size per station (MB)")
axes[2].set_xlabel("Station ID")
axes[2].set_ylabel("MB")
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


---
## 2. Core Segmentation Logic

Two-stage approach for each audio file:

1. **Speech removal** — invert the transcript timestamps to get noise intervals  
   (everything *between* speech segments, with a small padding on each edge).
2. **Silence removal** — within each noise interval, use `pydub.silence.detect_nonsilent`  
   to find stretches that are genuinely above `SILENCE_THRESH_DB` (-50 dBFS).  
   Any sub-interval that is silent for ≥ 1 s is dropped outright.  
   Only the truly active (noisy) sub-segments are saved.

In [ ]:
def get_noise_intervals(segments, audio_duration_s, padding_s=0.1):
    """
    Given a list of speech segments (each with 'start' and 'end' in seconds),
    return the complementary list of (start_s, end_s) tuples representing
    non-speech / noise portions of the audio.
    """
    if not segments:
        return [(0.0, audio_duration_s)]

    segs = sorted(segments, key=lambda s: s["start"])
    noise_intervals = []
    cursor = 0.0

    for seg in segs:
        noise_end = seg["start"] - padding_s
        if noise_end > cursor + padding_s:
            noise_intervals.append((cursor + padding_s, noise_end))
        cursor = seg["end"]

    # Trailing noise after last segment
    if cursor + padding_s < audio_duration_s:
        noise_intervals.append((cursor + padding_s, audio_duration_s))

    return noise_intervals


def remove_silence_from_chunk(chunk,
                               silence_thresh_db=SILENCE_THRESH_DB,
                               min_silence_len_ms=SILENCE_MIN_LEN_MS,
                               min_keep_ms=MIN_SEGMENT_MS):
    """
    Split a pydub AudioSegment into its non-silent sub-segments.

    Uses `detect_nonsilent` to find ranges where the audio is louder than
    `silence_thresh_db` (dBFS).  Stretches of continuous silence that last
    at least `min_silence_len_ms` ms are discarded.  Sub-segments shorter
    than `min_keep_ms` are also dropped.

    Returns a list of AudioSegment objects (may be empty).
    """
    nonsilent_ranges = detect_nonsilent(
        chunk,
        min_silence_len=min_silence_len_ms,
        silence_thresh=silence_thresh_db,
        seek_step=10,  # ms step for faster scanning
    )

    result = []
    for (start_ms, end_ms) in nonsilent_ranges:
        duration_ms = end_ms - start_ms
        if duration_ms >= min_keep_ms:
            result.append(chunk[start_ms:end_ms])
    return result


def segment_audio(mp3_path, json_path, output_dir,
                  min_segment_ms=MIN_SEGMENT_MS,
                  padding_s=0.1,
                  silence_thresh_db=SILENCE_THRESH_DB,
                  min_silence_len_ms=SILENCE_MIN_LEN_MS,
                  verbose=True):
    """
    Full pipeline for one audio file:
      1. Load MP3 and transcript JSON.
      2. Compute non-speech (noise) intervals.
      3. Within each interval, remove silent stretches (≥ min_silence_len_ms
         below silence_thresh_db).
      4. Save the remaining non-silent sub-segments as individual MP3 files
         under output_dir/<station>/<stem>/<stem>_N.mp3.

    Returns list of saved file paths.
    """
    # 1. Load audio
    audio = AudioSegment.from_mp3(str(mp3_path))
    duration_s = len(audio) / 1000.0

    # 2. Load transcript
    try:
        data = json.loads(json_path.read_text())
        if not isinstance(data, list) or len(data) == 0:
            if verbose:
                print(f"  [skip] {mp3_path.name}: empty/invalid JSON")
            return []
    except Exception as e:
        if verbose:
            print(f"  [skip] {mp3_path.name}: JSON error — {e}")
        return []

    # 3. Noise intervals (between speech)
    noise_intervals = get_noise_intervals(data, duration_s, padding_s=padding_s)

    stem       = mp3_path.stem
    out_subdir = output_dir / mp3_path.parent.name / stem
    out_subdir.mkdir(parents=True, exist_ok=True)

    # 4. Per-interval: remove silence, save non-silent sub-chunks
    saved  = []
    idx    = 1
    n_dropped_short   = 0
    n_dropped_silent  = 0

    for (t0, t1) in noise_intervals:
        interval_ms = int((t1 - t0) * 1000)
        if interval_ms < min_segment_ms:
            n_dropped_short += 1
            continue

        chunk = audio[int(t0 * 1000): int(t1 * 1000)]

        # Split chunk into non-silent sub-segments
        active_chunks = remove_silence_from_chunk(
            chunk,
            silence_thresh_db=silence_thresh_db,
            min_silence_len_ms=min_silence_len_ms,
            min_keep_ms=min_segment_ms,
        )

        if not active_chunks:
            n_dropped_silent += 1
            continue

        for sub in active_chunks:
            out_path = out_subdir / f"{stem}_{idx}.mp3"
            sub.export(str(out_path), format="mp3")
            saved.append(out_path)
            idx += 1

    if verbose:
        print(
            f"  {mp3_path.name}: "
            f"{len(noise_intervals)} noise intervals → "
            f"{n_dropped_short} too-short, "
            f"{n_dropped_silent} all-silent, "
            f"{len(saved)} saved"
        )

    return saved


print("Helper functions defined.")

---
## 3. Smoke-test on a Single File

Run one file end-to-end, inspect the intervals, listen to the result.

In [ ]:
# Pick first file (df already contains only transcript-paired files)
test_row  = df.iloc[0]
test_mp3  = test_row["mp3_path"]
test_json = test_row["json_path"]

print(f"Test file : {test_mp3}")
print(f"Test JSON : {test_json}")

segments = json.loads(test_json.read_text())
print(f"\nTranscript: {len(segments)} speech segments")
for s in segments[:5]:
    print(f"  [{s['start']:.2f}s – {s['end']:.2f}s]  {s.get('text','').strip()}")
if len(segments) > 5:
    print(f"  ... and {len(segments) - 5} more")


In [ ]:
# Show the computed noise intervals before saving anything
test_audio    = AudioSegment.from_mp3(str(test_mp3))
duration_s    = len(test_audio) / 1000.0
noise_intervals = get_noise_intervals(segments, duration_s, padding_s=0.1)

print(f"Audio duration : {duration_s:.2f}s")
print(f"Noise intervals: {len(noise_intervals)}")
for i, (t0, t1) in enumerate(noise_intervals[:10], 1):
    print(f"  {i:3d}. [{t0:.2f}s – {t1:.2f}s]  ({(t1-t0)*1000:.0f} ms)")
if len(noise_intervals) > 10:
    print(f"  ... and {len(noise_intervals) - 10} more")

In [ ]:
# Actually segment and save the test file
TEST_OUTPUT = Path("output_segmented_test")

saved_paths = segment_audio(
    mp3_path       = test_mp3,
    json_path      = test_json,
    output_dir     = TEST_OUTPUT,
    min_segment_ms = MIN_SEGMENT_MS,
    padding_s      = 0.1,
    verbose        = True,
)

print(f"\nSaved {len(saved_paths)} segments to: {TEST_OUTPUT.resolve()}")
for p in saved_paths[:8]:
    print(f"  {p.name}  ({p.stat().st_size/1024:.1f} KB)")
if len(saved_paths) > 8:
    print(f"  ... and {len(saved_paths) - 8} more files")

In [ ]:
# Waveform + playback of the first test segment
from IPython.display import Audio, display
import numpy as np

if saved_paths:
    sample_path = saved_paths[0]
    seg_audio = AudioSegment.from_mp3(str(sample_path))
    samples = np.array(seg_audio.get_array_of_samples(), dtype=np.float32)
    if seg_audio.channels == 2:
        samples = samples.reshape(-1, 2).mean(axis=1)
    sr = seg_audio.frame_rate
    t  = np.linspace(0, len(samples) / sr, len(samples))

    plt.figure(figsize=(12, 2))
    plt.plot(t, samples, linewidth=0.4, color="teal")
    plt.title(f"Waveform — {sample_path.name}  ({len(samples)/sr:.2f}s)")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

    display(Audio(samples, rate=sr))
else:
    print("No segments were saved — check the test file.")

---
## 4. Full Dataset Processing

Only files with a **valid, non-empty** JSON transcript are processed.  
Output structure:
```
output_segmented/
  <station>/
    <stem>/
      <stem>_1.mp3
      <stem>_2.mp3
      ...
```

In [ ]:
# Build the processing list — every row in df has a valid transcript
to_process = [
    (row["mp3_path"], row["json_path"])
    for _, row in df.iterrows()
]

print(f"Files to process : {len(to_process)}  (all have valid transcripts)")
print(f"Output directory : {OUTPUT_DIR.resolve()}")


In [ ]:
results = []

for mp3_path, json_path in tqdm(to_process, desc="Segmenting", unit="file"):
    saved = segment_audio(
        mp3_path       = mp3_path,
        json_path      = json_path,
        output_dir     = OUTPUT_DIR,
        min_segment_ms = MIN_SEGMENT_MS,
        padding_s      = 0.1,
        verbose        = False,   # set True for per-file logs
    )
    results.append((mp3_path.name, len(saved)))

print("Done!")
print(f"Total segments saved: {sum(n for _, n in results)}")

In [ ]:
# Summary
result_df = pd.DataFrame(results, columns=["file", "segments_saved"])
result_df = result_df.sort_values("segments_saved", ascending=False)

print(f"Files processed  : {len(result_df)}")
print(f"Total segments   : {result_df['segments_saved'].sum()}")
print(f"Avg per file     : {result_df['segments_saved'].mean():.1f}")
print(f"Min / Max        : {result_df['segments_saved'].min()} / {result_df['segments_saved'].max()}")
print()
print(result_df.to_string(index=False))

In [ ]:
# Verify output on disk
all_output_files = list(OUTPUT_DIR.rglob("*.mp3"))
total_size_mb = sum(f.stat().st_size for f in all_output_files) / (1024**2)
print(f"Output segments: {len(all_output_files)} files, {total_size_mb:.1f} MB total")

for f in sorted(all_output_files)[:10]:
    print(f"  output_segmented/{f.relative_to(OUTPUT_DIR)}")
if len(all_output_files) > 10:
    print(f"  ... and {len(all_output_files) - 10} more")

# this is shit

I have watched the audio, I see that they tagged this with whisper, and it missed al ot of places where there is noise and speech in the same time,
because we don't want speech in our noise corpus, I have manually picked over about 7 frequencies, the noises that looked okay and didn't contain any speech.

now I want to see a bit of the noises, what the original frequncies, do konw if I show resamples them to remove signal that we wouldn't originall have (since our real data is in 8khz)

In [1]:
import os

target_dir = "/Users/Roi/Desktop/asr-training/audio-set"

# Change the current working directory
os.chdir(target_dir)

# Verify the change
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/Roi/Desktop/asr-training/audio-set


In [6]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from mutagen.mp3 import MP3

input_path: Path = Path("output_")

# Looking specifically for mp3s now
audio_extensions = {'.mp3'} 
audio_metadata = []

print("Scanning directory for MP3 files...")
# Gather files into a list first so tqdm knows the total count
audio_files = [
    p for p in input_path.rglob('*') 
    if p.is_file() and p.suffix.lower() in audio_extensions
]

print(f"Found {len(audio_files)} audio files. Starting extraction...")

# Wrap the list in tqdm
for file_path in tqdm(audio_files, desc="Extracting Metadata", unit="file"):
    try:
        # Load the MP3 metadata
        audio = MP3(file_path)
        
        audio_metadata.append({
            "filename": file_path.name,
            "filepath": str(file_path),
            "sample_rate": audio.info.sample_rate,
            "channels": audio.info.channels,
            "duration_sec": audio.info.length,
            "bitrate": audio.info.bitrate,
            "format": "mp3"
        })
    except Exception as e:
        # Use tqdm.write so it doesn't break the progress bar visual
        tqdm.write(f"Could not read {file_path.name}: {e}")

# Convert the list of dictionaries into a DataFrame
df = pd.DataFrame(audio_metadata)

print("\nFinished!")
df.head(10)

Scanning directory for MP3 files...
Found 56 audio files. Starting extraction...


Extracting Metadata: 100%|██████████| 56/56 [00:00<00:00, 9197.31file/s]


Finished!


,filename,filepath,sample_rate,channels,duration_sec,bitrate,format
0,37404-20240102-0641_7.mp3,output_/37404-20240102-0641_7.mp3,22050,1,1.097143,32003,mp3
1,37640-20231011-0618_24.mp3,output_/37640-20231011-0618_24.mp3,44100,1,0.574694,63992,mp3
2,17329-20230709-1317_2.mp3,output_/17329-20230709-1317_2.mp3,22050,1,0.757551,31998,mp3
3,37404-20240102-0641_4.mp3,output_/37404-20240102-0641_4.mp3,22050,1,5.355102,31999,mp3
4,17329-20230709-1317_3.mp3,output_/17329-20230709-1317_3.mp3,22050,1,0.835918,32003,mp3
5,37640-20230819-0814_13.mp3,output_/37640-20230819-0814_13.mp3,44100,1,2.768980,63998,mp3
6,37640-20231011-0042_4.mp3,output_/37640-20231011-0042_4.mp3,44100,1,0.653061,63994,mp3
7,26383-20231221-2129_1.mp3,output_/26383-20231221-2129_1.mp3,22050,1,1.567347,31998,mp3
8,30659-20231101-0813_5.mp3,output_/30659-20231101-0813_5.mp3,8000,1,2.304000,8000,mp3
9,37640-20231011-0042_5.mp3,output_/37640-20231011-0042_5.mp3,44100,1,1.567347,63996,mp3


In [9]:
from pathlib import Path
import librosa
import soundfile as sf
from tqdm import tqdm

input_path: Path = Path("output_")
# Define and create the new directory
output_path: Path = Path("noise_input_16khz")
output_path.mkdir(parents=True, exist_ok=True)

# Find all mp3 files
audio_files = [p for p in input_path.rglob('*') if p.is_file() and p.suffix.lower() == '.mp3']

print(f"Found {len(audio_files)} files. Starting processing...")

for file_path in tqdm(audio_files, desc="Processing Audio", unit="file"):
    try:
        # We change the extension of the output file to .wav
        new_file_name = file_path.with_suffix('.wav').name
        new_file_path = output_path / new_file_name
        
        # Skip if we already processed this file (useful if the script gets interrupted)
        if new_file_path.exists():
            continue

        # 1. Load the audio file (sr=None preserves the original sample rate)
        y, original_sr = librosa.load(file_path, sr=None)
        
        # 2. Downsample to 8kHz (simulating low bandwidth)
        y_8k = librosa.resample(y, orig_sr=original_sr, target_sr=8000)
        
        # 3. Upsample back to 16kHz (matching standard ASR input formats)
        y_16k = librosa.resample(y_8k, orig_sr=8000, target_sr=16000)
        
        # 4. Save the result as a .wav file
        sf.write(new_file_path, y_16k, 16000)
        
    except Exception as e:
        tqdm.write(f"Error processing {file_path.name}: {e}")

print(f"\nFinished! All files are saved in: {output_path.absolute()}")

Found 56 files. Starting processing...


Processing Audio: 100%|██████████| 56/56 [00:00<00:00, 1053.15file/s]


Finished! All files are saved in: /Users/Roi/Desktop/asr-training/audio-set/noise_input_16khz


In [8]:
from collections import Counter
import pandas as pd
from pathlib import Path

# Setup the path and grab the files (if you don't already have the audio_files list in memory)
input_path = Path("output_")
audio_files = [p for p in input_path.rglob('*') if p.is_file() and p.suffix.lower() == '.mp3']

# Extract the prefix: split the filename by '-' and take the first part [0]
prefixes = [file_path.name.split('-')[0] for file_path in audio_files]

# Count how many times each prefix appears
bucket_counts = Counter(prefixes)

# Convert the counts into a pandas DataFrame for a clean table view
df_buckets = pd.DataFrame(bucket_counts.items(), columns=['Prefix', 'Count'])

# Sort it so the prefixes with the most files are at the top
df_buckets = df_buckets.sort_values(by='Count', ascending=False).reset_index(drop=True)

print(f"Total unique prefixes: {len(df_buckets)}")
print(df_buckets)

Total unique prefixes: 11
   Prefix  Count
0   37640     11
1   26694     10
2   17329      7
3   26383      7
4   22612      6
5   37404      4
6   31445      4
7   30659      3
8   37460      2
9   22851      1
10  38117      1
